# QCar AttFuse — Training Walkthrough

This notebook follows `CP_Fusion_MODELS.ipynb`, but completes the full training path for the calibrated **front-only** QCar dataset. It deliberately uses `OnlyFront_dev`; there is no same-trajectory test set.

**Pipeline:** images → LSS encoder → BEV backbone → shrinker → affine warp + AttFusion → post-fusion heads → detection loss → backward → validation.

## Phase 0 — Environment and reproducibility
Run Jupyter from the HEAL repository or its `qcar_notebooks/` directory. Full training requires CUDA.

In [1]:
from pathlib import Path
from types import SimpleNamespace
import os, random, statistics, json, gc, copy
from datetime import datetime
import numpy as np
import torch
from torch.utils.data import DataLoader

HEAL_ROOT = Path.cwd()
if not (HEAL_ROOT / 'opencood').is_dir():
    HEAL_ROOT = (HEAL_ROOT / '..').resolve()
os.chdir(HEAL_ROOT)

import qcar.patches.patch_1cam_loader  # activates front-only discovery
import opencood.hypes_yaml.yaml_utils as yaml_utils
from opencood.data_utils.datasets import build_dataset
from opencood.tools import train_utils

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), 'Full HEAL training requires a CUDA GPU.'
DEVICE = torch.device('cuda')
print('HEAL root:', HEAL_ROOT)
print('Device:', DEVICE)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[patch_1cam_loader] OPV2VBaseDataset.find_camera_files -> 1-camera (front only)
HEAL root: /mnt/mainvolume/Backup/Projects/HEAL
Device: cuda


## Phase 1 — Load and inspect the resolved experiment configuration
The YAML controls the dataset factory, LSS geometry, AttFusion, post-fusion heads, loss, optimizer, and thresholds.

In [2]:
CONFIG_PATH = 'qcar/configs/camera_attfuse_onlyfront.yaml'
hypes = yaml_utils.load_yaml(CONFIG_PATH, SimpleNamespace(model_dir=''))
hypes['train_params']['batch_size'] = 1  # verified for the available 3.68 GiB GPU

assert hypes['root_dir'].endswith('OnlyFront_dev/train')
assert hypes['validate_dir'].endswith('OnlyFront_dev/validate')
assert hypes.get('test_dir') is None
assert hypes['model']['args']['fusion_method'] == 'att'
print('Model:', hypes['model']['core_method'])
print('Train:', hypes['root_dir'])
print('Validate:', hypes['validate_dir'])
print('Checkpoint:', hypes['_qcar_pretrained_checkpoint'])
print('Image geometry:', hypes['heter']['modality_setting']['m2']['data_aug_conf'])

Model: heter_model_baseline
Train: qcar_dataset/Inference/OnlyFront_dev/train
Validate: qcar_dataset/Inference/OnlyFront_dev/validate
Checkpoint: checkpoints/opv2v_camera/HeterBaseline_opv2v_camera_attfuse_2023_08_08_16_50_01/net_epoch_bestval_at17.pth
Image geometry: {'resize_lim': [0.8125, 0.875], 'final_dim': [384, 512], 'rot_lim': [-3.6, 3.6], 'H': 480, 'W': 640, 'rand_flip': False, 'bot_pct_lim': [0.0, 0.05], 'cams': ['camera0', 'camera1', 'camera2', 'camera3'], 'Ncams': 1}


## Phase 2 — Build train and validation loaders
Negative-only batches are retained: they provide classification supervision against false positives. Validation is 10 positive + 10 negative frames.

In [3]:
train_dataset = build_dataset(hypes, visualize=False, train=True)
val_dataset = build_dataset(hypes, visualize=False, train=False)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0,
                          collate_fn=train_dataset.collate_batch_train,
                          pin_memory=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0,
                        collate_fn=val_dataset.collate_batch_train,
                        pin_memory=True, drop_last=False)
val_presence = [int(val_dataset[i]['ego']['object_bbx_mask'].sum()) > 0
                for i in range(len(val_dataset))]
print('Train frames:', len(train_dataset), 'batches:', len(train_loader))
print('Validation frames:', len(val_dataset),
      'positive:', sum(val_presence), 'negative:', len(val_presence)-sum(val_presence))

Dataset dir: qcar_dataset/Inference/OnlyFront_dev/train
len: 73
len: 73
Dataset dir: qcar_dataset/Inference/OnlyFront_dev/validate
len: 20
len: 20
Train frames: 73 batches: 37
Validation frames: 20 positive: 10 negative: 10


## Phase 3 — Instantiate encoder, fusion, and post-fusion modules
`create_model` constructs the same modules used by HEAL training. We then load the official OPV2V AttFuse checkpoint strictly (530/530 tensors in the verified setup).

In [4]:
model = train_utils.create_model(hypes)
criterion = train_utils.create_loss(hypes)
checkpoint_path = HEAL_ROOT / hypes['_qcar_pretrained_checkpoint']
checkpoint = torch.load(checkpoint_path, map_location='cpu')
initial_state = checkpoint.get('model_state_dict', checkpoint)
model.load_state_dict(initial_state, strict=True)
model = model.to(DEVICE)
optimizer = train_utils.setup_optimizer(hypes, model)
scheduler = train_utils.setup_lr_schedular(hypes, optimizer)
USE_AMP = True
GRADIENT_ACCUMULATION_STEPS = 2  # effective batch size 2 with batch_size=1
GRADIENT_CLIP_NORM = 10.0
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

encoder = model.encoder_m2
bev_backbone = model.backbone_m2
shrinker = model.shrinker_m2
fusion = model.fusion_net
post_fusion = {'classification': model.cls_head,
               'regression': model.reg_head,
               'direction': model.dir_head}
print('Encoder:', type(encoder).__name__)
print('BEV backbone:', type(bev_backbone).__name__)
print('Pre-fusion shrinker:', type(shrinker).__name__)
print('Fusion:', type(fusion).__name__)
print('Post-fusion (heads directly after AttFusion):',
      {k: type(v).__name__ for k, v in post_fusion.items()})

/home/saavkd7/.pyenv/versions/heal38/lib/python3.8/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loaded pretrained weights for efficientnet-b0
=========Those modules have trainable component=========
encoder_m2
backbone_m2
shrinker_m2
cls_head
reg_head
dir_head

=========Those modules have untrainable component=========


Encoder: LiftSplatShoot
BEV backbone: BaseBEVBackbone
Pre-fusion shrinker: DownsampleConv
Fusion: AttFusion
Post-fusion (heads directly after AttFusion): {'classification': 'Conv2d', 'regression': 'Conv2d', 'direction': 'Conv2d'}


## Phase 4 — Trace one complete forward pass
AttFusion first warps peer BEV features into the ego frame using normalized affine matrices. At every BEV cell it applies scaled dot-product attention across agents. In this configuration, the fused ego feature passes directly into the classification, regression, and direction heads; there is no separate post-fusion shrinker.

In [5]:
def summarize(value):
    if torch.is_tensor(value): return tuple(value.shape)
    if isinstance(value, dict): return {k: summarize(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)): return [summarize(v) for v in value]
    return type(value).__name__

captured = {}
handles = []
for name, module in [('LSS encoder', encoder), ('BEV backbone', bev_backbone),
                     ('pre-fusion shrinker', shrinker), ('AttFusion', fusion),
                     ('post-fusion classification', model.cls_head),
                     ('post-fusion regression', model.reg_head),
                     ('post-fusion direction', model.dir_head)]:
    handles.append(module.register_forward_hook(
        lambda mod, inputs, output, name=name: captured.update({name: summarize(output)})))

batch = next(iter(train_loader))
batch = train_utils.to_device(batch, DEVICE)
model.eval()
with torch.inference_mode(), torch.cuda.amp.autocast(enabled=USE_AMP):
    output = model(batch['ego'])
for handle in handles: handle.remove()
print('record_len:', batch['ego']['record_len'].tolist())
print('pairwise transform:', tuple(batch['ego']['pairwise_t_matrix'].shape))
for stage, shape in captured.items(): print(stage, '->', shape)
print('Final outputs:', {k: tuple(v.shape) for k, v in output.items() if torch.is_tensor(v)})
del output, batch, captured
gc.collect(); torch.cuda.empty_cache()

record_len: [2, 2]
pairwise transform: (2, 5, 5, 4, 4)
LSS encoder -> (4, 128, 256, 256)
BEV backbone -> {'spatial_features': (4, 128, 256, 256), 'spatial_features_2d': (4, 384, 128, 128)}
pre-fusion shrinker -> (4, 256, 128, 128)
AttFusion -> (2, 256, 128, 128)
post-fusion classification -> (2, 2, 128, 128)
post-fusion regression -> (2, 14, 128, 128)
post-fusion direction -> (2, 4, 128, 128)
Final outputs: {'cls_preds': (2, 2, 128, 128), 'reg_preds': (2, 14, 128, 128), 'dir_preds': (2, 4, 128, 128)}


## Phase 5 — One safe optimization-step walkthrough
The default computes forward, AMP-scaled loss, and backward but does **not** update weights. The earlier batch-size-2 OOM output is intentionally preserved; restart the kernel and rerun from Phase 0 so the new batch-size-1 loaders replace the old objects.

In [6]:
APPLY_OPTIMIZER_STEP = False
torch.cuda.empty_cache()
batch = train_utils.to_device(next(iter(train_loader)), DEVICE)
model.train(); optimizer.zero_grad(set_to_none=True); model.zero_grad(set_to_none=True)
with torch.cuda.amp.autocast(enabled=USE_AMP):
    output = model(batch['ego'])
    loss = criterion(output, batch['ego']['label_dict'])
scaler.scale(loss).backward()
gradients_finite = all(torch.isfinite(p.grad).all().item()
                       for p in model.parameters() if p.grad is not None)
if APPLY_OPTIMIZER_STEP:
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
    scaler.step(optimizer); scaler.update()
print('Loss:', float(loss.detach()), 'finite gradients:', gradients_finite,
      'weights updated:', APPLY_OPTIMIZER_STEP)
optimizer.zero_grad(set_to_none=True); model.zero_grad(set_to_none=True)
del output, loss, batch
gc.collect(); torch.cuda.empty_cache()

RuntimeError: CUDA out of memory. Tried to allocate 254.00 MiB (GPU 0; 3.68 GiB total capacity; 2.17 GiB already allocated; 280.88 MiB free; 2.26 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

## Phase 6 — Validation function and controlled training loop
Every validation batch is used, including negative-only batches. The complete loop below supports AMP, gradient accumulation, clipping, resumable state, raw HEAL-compatible best checkpoints, and JSON history.

In [ ]:
def validate(model, loader):
    model.eval(); losses = []
    with torch.inference_mode():
        for val_batch in loader:
            val_batch = train_utils.to_device(val_batch, DEVICE)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                val_output = model(val_batch['ego'])
                val_loss = criterion(val_output, val_batch['ego']['label_dict'])
            losses.append(float(val_loss))
    return statistics.mean(losses)

baseline_val_loss = validate(model, val_loader)
print('Baseline validation loss:', baseline_val_loss)

In [ ]:
RUN_FULL_TRAINING = False  # change to True only after restarting and rerunning all cells
EPOCHS = hypes['train_params']['epoches']
RESUME_STATE = None  # set to SAVE_DIR/'training_state_last.pth' to resume
RUN_NAME = 'qcar_attfuse_' + datetime.now().strftime('%Y%m%d_%H%M%S')
SAVE_DIR = (Path(RESUME_STATE).parent if RESUME_STATE is not None
            else HEAL_ROOT / 'opencood/logs' / RUN_NAME)
if RUN_FULL_TRAINING:
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    with open(SAVE_DIR/'resolved_hypes.json', 'w') as stream:
        json.dump(hypes, stream, indent=2, default=str)
    best, start_epoch, history = float('inf'), 0, []
    if RESUME_STATE is not None:
        saved = torch.load(RESUME_STATE, map_location='cpu')
        model.load_state_dict(saved['model_state_dict'], strict=True)
        optimizer.load_state_dict(saved['optimizer_state_dict'])
        scheduler.load_state_dict(saved['scheduler_state_dict'])
        scaler.load_state_dict(saved['scaler_state_dict'])
        best = saved['best_val_loss']; start_epoch = saved['epoch'] + 1
        history = saved.get('history', [])
    for epoch in range(start_epoch, EPOCHS):
        model.train(); train_losses = []
        optimizer.zero_grad(set_to_none=True); model.zero_grad(set_to_none=True)
        for step, train_batch in enumerate(train_loader):
            train_batch = train_utils.to_device(train_batch, DEVICE)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                train_output = model(train_batch['ego'])
                full_loss = criterion(train_output, train_batch['ego']['label_dict'])
                scaled_loss = full_loss / GRADIENT_ACCUMULATION_STEPS
            scaler.scale(scaled_loss).backward()
            update_now = ((step + 1) % GRADIENT_ACCUMULATION_STEPS == 0
                          or step + 1 == len(train_loader))
            if update_now:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True); model.zero_grad(set_to_none=True)
            train_losses.append(float(full_loss.detach()))
            del train_output, full_loss, scaled_loss, train_batch
        val_loss = validate(model, val_loader)
        train_loss = statistics.mean(train_losses)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
        print(f'epoch {epoch}: train={train_loss:.6f} val={val_loss:.6f}')
        if val_loss < best:
            best = val_loss
            for old_best in SAVE_DIR.glob('net_epoch_bestval_at*.pth'):
                old_best.unlink()
            torch.save(model.state_dict(), SAVE_DIR/f'net_epoch_bestval_at{epoch+1}.pth')
        scheduler.step(epoch)
        training_state = {'epoch': epoch, 'model_state_dict': model.state_dict(),
                 'optimizer_state_dict': optimizer.state_dict(),
                 'scheduler_state_dict': scheduler.state_dict(),
                 'scaler_state_dict': scaler.state_dict(),
                 'best_val_loss': best, 'history': history}
        torch.save(training_state, SAVE_DIR/'training_state_last.pth')
        with open(SAVE_DIR/'training_history.json', 'w') as stream:
            json.dump(history, stream, indent=2)
        gc.collect(); torch.cuda.empty_cache()
    print('Training complete:', SAVE_DIR)
else:
    print('Production-style training is ready. Set RUN_FULL_TRAINING=True to run it.')

## Phase 7 — Final fit on all 113 development frames
After choosing the epoch count from development validation, restart from the official initialization and fit once on `OnlyFront_dev/final_train`. This phase performs no model selection.

In [ ]:
RUN_FINAL_FIT = False
SELECTED_FINAL_EPOCHS = None  # freeze this integer from the development run
if RUN_FINAL_FIT:
    assert isinstance(SELECTED_FINAL_EPOCHS, int) and SELECTED_FINAL_EPOCHS > 0
    final_hypes = copy.deepcopy(hypes)
    final_hypes['root_dir'] = hypes['_qcar_final_train_dir']
    final_dataset = build_dataset(final_hypes, visualize=False, train=True)
    final_loader = DataLoader(final_dataset, batch_size=1, shuffle=True, num_workers=0,
                              collate_fn=final_dataset.collate_batch_train,
                              pin_memory=True, drop_last=False)
    final_model = train_utils.create_model(final_hypes)
    final_model.load_state_dict(initial_state, strict=True)
    final_model = final_model.to(DEVICE)
    final_criterion = train_utils.create_loss(final_hypes)
    final_optimizer = train_utils.setup_optimizer(final_hypes, final_model)
    final_scheduler = train_utils.setup_lr_schedular(final_hypes, final_optimizer)
    final_scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    final_dir = HEAL_ROOT/'opencood/logs'/('qcar_attfuse_final_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
    final_dir.mkdir(parents=True, exist_ok=False)
    for epoch in range(SELECTED_FINAL_EPOCHS):
        final_model.train(); final_optimizer.zero_grad(set_to_none=True)
        for step, item in enumerate(final_loader):
            item = train_utils.to_device(item, DEVICE)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                prediction = final_model(item['ego'])
                final_loss = final_criterion(prediction, item['ego']['label_dict'])
                accumulation_loss = final_loss / GRADIENT_ACCUMULATION_STEPS
            final_scaler.scale(accumulation_loss).backward()
            update_now = ((step + 1) % GRADIENT_ACCUMULATION_STEPS == 0
                          or step + 1 == len(final_loader))
            if update_now:
                final_scaler.unscale_(final_optimizer)
                torch.nn.utils.clip_grad_norm_(final_model.parameters(), GRADIENT_CLIP_NORM)
                final_scaler.step(final_optimizer); final_scaler.update()
                final_optimizer.zero_grad(set_to_none=True)
        final_scheduler.step(epoch)
        print('final-fit epoch', epoch, 'loss', float(final_loss.detach()))
    final_checkpoint = final_dir/f'net_epoch{SELECTED_FINAL_EPOCHS}.pth'
    torch.save(final_model.state_dict(), final_checkpoint)
    with open(final_dir/'final_fit_protocol.json', 'w') as stream:
        json.dump({'frames': len(final_dataset), 'epochs': SELECTED_FINAL_EPOCHS,
                   'seed': SEED, 'checkpoint': str(final_checkpoint),
                   'test_policy': 'new independent trajectory'}, stream, indent=2)
    print('Final checkpoint:', final_checkpoint)
else:
    print('Final fit disabled until SELECTED_FINAL_EPOCHS is frozen.')

In [ ]:
# Strict handoff check for the inference notebook.
if RUN_FINAL_FIT:
    handoff_model = train_utils.create_model(final_hypes)
    handoff_model.load_state_dict(torch.load(final_checkpoint, map_location='cpu'), strict=True)
    print('Inference handoff checkpoint loads strictly: PASS')

## Production boundary
The development best checkpoint is for analysis and epoch selection. The final-fit checkpoint uses all 113 frames and must be evaluated only on a newly collected independent trajectory using `QCar_AttFuse_Inference.ipynb`.